In [ ]:
from triadExtractor import TriadExtractor
import os
import json

def run_triad_extractor_comparison(full_matches):
    """Compare TriadExtractor method against iReal ground truth, saving tonality in triad_chords JSON."""
    extractor = TriadExtractor(hop_length=512)
    results = []
    
    # Create triad_references directory if it doesn't exist
    triad_references_dir = "../dataset/triad_references"
    os.makedirs(triad_references_dir, exist_ok=True)
    
    for i, match in enumerate(full_matches):
        song_id = match['lastfm_id']
        song_name = match['lastfm_song_name']
        print(f"Processing {i+1}/{len(full_matches)}: {song_name}")
        
        try:
            # Load audio file
            audio_path = f"/mnt/Aimir_HD/lastfm/audio/{song_id}.mp3"
            if not os.path.exists(audio_path):
                print(f"  Audio file not found: {audio_path}")
                continue
            
            # Extract chords using TriadExtractor
            print(f"  Extracting chords from: {audio_path}")
            chord_changes = extractor.extract_chords(
                audio_path,
                threshold=0.3,
                check_on_beat=True
            )
            
            # Convert to format compatible with compare_song_chords
            triad_chords = []
            for chord_change in chord_changes:
                triad_chords.append({
                    'chord_name': chord_change.chord,
                    'weight': 1,  # Equal weight for all chords
                    'timestamp': chord_change.timestamp
                })
            
            # Create song-specific directory
            song_dir = os.path.join(triad_references_dir, song_id)
            os.makedirs(song_dir, exist_ok=True)
            
            # --- Get tonality from iReal ---
            ireal_file = match['ireal_file']
            ireal_path = os.path.join(ireal_dir, ireal_file)
            with open(ireal_path, 'r') as f:
                ireal_data = json.load(f)
            tonality = ireal_data.get("metadata", {}).get("tonality", None)
            
            # --- Save triad_chords and tonality to JSON file ---
            triad_json_obj = {
                "tonality": tonality,
                "chords": triad_chords
            }
            triad_json_path = os.path.join(song_dir, f"{song_id}_triad_chords.json")
            with open(triad_json_path, 'w') as f:
                json.dump(triad_json_obj, f, indent=2)
            
            print(f"  Saved triad chords to: {triad_json_path}")
            
            # Run comparison
            result = compare_song_chords(ireal_data, triad_chords)
            print(f"  TriadExtractor Score: {result['avg_similarity']:.3f}")
            print("-" * 50)
            
            results.append({
                'song_id': song_id,
                'song_name': song_name,
                'triad_score': result['avg_similarity'],
                'triad_vocab_overlap': result['vocabulary_overlap'],
                'ireal_vocab_size': result['ireal_vocab_size'],
                'triad_vocab_size': result['lastfm_vocab_size']
            })
            
        except Exception as e:
            print(f"  Error processing {song_name}: {e}")
            continue
    
    return results


# Run the TriadExtractor comparison
print("=== RUNNING TRIAD EXTRACTOR COMPARISON ===")
triad_results = run_triad_extractor_comparison(full_matches)

# Show results
df_triad = pd.DataFrame(triad_results)
df_triad_sorted = df_triad.sort_values('triad_score', ascending=False)

print("\n=== TRIAD EXTRACTOR SCORES PER SONG ===")
for _, row in df_triad_sorted.iterrows():
    score = row['triad_score']
    status = "EXCELLENT" if score > 0.9 else "GOOD" if score > 0.8 else "ACCEPTABLE" if score > 0.6 else "BAD"
    print(f"{row['song_name'][:40]:40} | {score:.3f} | {status}")

print(f"\nTriadExtractor Average: {df_triad['triad_score'].mean():.3f}")